Total running time: about 45 min.

In [6]:
import os
from glob import glob
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display
import cv2
from tqdm import tqdm
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

## Collect the data and split train set vs test set

In [7]:
# # Paths adjusted for notebooks folder structure
# PROJECT_ROOT = os.path.join(os.getcwd(), '..')  # Go up one level from notebooks/
# INPUT_PATH = os.path.join(PROJECT_ROOT, 'data')
# OUTPUT_PATH = os.path.join(PROJECT_ROOT, 'data_processed')
# os.makedirs(OUTPUT_PATH, exist_ok=True)
# TRAIN_ANNOTATIONS_DIR = os.path.join(INPUT_PATH, 'DETRAC-Train-Annotations-XML', 'DETRAC-Train-Annotations-XML')
# TEST_ANNOTATIONS_DIR = os.path.join(INPUT_PATH, 'DETRAC-Test-Annotations-XML', 'DETRAC-Test-Annotations-XML')
# ALL_IMAGES_DIR = os.path.join(INPUT_PATH, 'DETRAC-Images', 'DETRAC-Images')

In [8]:
# Paths adjusted for notebooks folder structure
PROJECT_ROOT = Path.cwd().parent  # one level up from notebooks/
INPUT_PATH = PROJECT_ROOT / "data"
OUTPUT_PATH = PROJECT_ROOT / "data_processed"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
# Annotations (dossiers réellement imbriqués)
TRAIN_ANNOTATIONS_DIR = INPUT_PATH / "DETRAC-Train-Annotations-XML" / "DETRAC-Train-Annotations-XML"
TEST_ANNOTATIONS_DIR = INPUT_PATH / "DETRAC-Test-Annotations-XML" / "DETRAC-Test-Annotations-XML"
# Images (dossier réellement imbriqué)
ALL_IMAGES_DIR = INPUT_PATH / "DETRAC-Images" / "DETRAC-Images"

In [ ]:
# --- Vérification des dossiers racine ---
assert os.path.isdir(TRAIN_ANNOTATIONS_DIR), f"TRAIN_ANNOTATIONS_DIR not found: {TRAIN_ANNOTATIONS_DIR}"
assert os.path.isdir(TEST_ANNOTATIONS_DIR), f"TEST_ANNOTATIONS_DIR not found: {TEST_ANNOTATIONS_DIR}"
assert os.path.isdir(ALL_IMAGES_DIR), f"ALL_IMAGES_DIR not found: {ALL_IMAGES_DIR}"
assert os.path.isdir(OUTPUT_PATH), f"OUTPUT_PATH not found: {OUTPUT_PATH}"

# Split the dataset into training and testing sets based on annotations
train_annotation_files = glob(os.path.join(TRAIN_ANNOTATIONS_DIR, '*.xml'))
test_annotation_files = glob(os.path.join(TEST_ANNOTATIONS_DIR, '*.xml'))

if len(train_annotation_files) == 0:
    raise RuntimeError(f"No training annotation files found in {TRAIN_ANNOTATIONS_DIR}")

if len(test_annotation_files) == 0:
    raise RuntimeError(f"No testing annotation files found in {TEST_ANNOTATIONS_DIR}")

train_image_files = []
test_image_files = []

for ann_file in train_annotation_files:
    # Use filename as sequence name instead of parsing XML
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    img_dir = os.path.join(ALL_IMAGES_DIR, seq_name)

    if not os.path.isdir(img_dir):
        print(f"Warning: Image directory not found for sequence '{seq_name}' at {img_dir}")
        continue

    img_files = sorted(glob(os.path.join(img_dir, '*.jpg')))
    print(f"Found {len(img_files)} images in {seq_name}")
    train_image_files.extend(img_files)

for ann_file in test_annotation_files:
    # Use filename as sequence name instead of parsing XML
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    img_dir = os.path.join(ALL_IMAGES_DIR, seq_name)

    if not os.path.isdir(img_dir):
        print(f"Warning: Image directory not found for sequence '{seq_name}' at {img_dir}")
        continue

    img_files = sorted(glob(os.path.join(img_dir, '*.jpg')))
    print(f"Found {len(img_files)} images in {seq_name}")
    test_image_files.extend(img_files)

# Copy images to processed directory preserving sequence structure
train_output_dir = os.path.join(OUTPUT_PATH, 'train')
test_output_dir = os.path.join(OUTPUT_PATH, 'test')

assert os.path.isdir(train_output_dir), f"train_output_dir not found: {train_output_dir}"
assert os.path.isdir(test_output_dir), f"test_output_dir not found: {test_output_dir}"


os.makedirs(train_output_dir, exist_ok=True)
os.makedirs(test_output_dir, exist_ok=True)

# Copy training images with directory structure
for ann_file in tqdm(train_annotation_files, desc='Copying training sequences'):
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    source_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    dest_dir = os.path.join(train_output_dir, seq_name)

    if not os.path.isdir(source_dir):
        print(f"Warning: Source directory missing, skipped: {source_dir}")
        continue

    os.makedirs(dest_dir, exist_ok=True)
    img_files = sorted(glob(os.path.join(source_dir, '*.jpg')))

    for img_file in img_files:
        dst = os.path.join(dest_dir, os.path.basename(img_file))

        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(img_file):
            continue

        shutil.copy(img_file, dst)

# Copy testing images with directory structure
for ann_file in tqdm(test_annotation_files, desc='Copying testing sequences'):
    seq_name = os.path.splitext(os.path.basename(ann_file))[0]
    source_dir = os.path.join(ALL_IMAGES_DIR, seq_name)
    dest_dir = os.path.join(test_output_dir, seq_name)

    if not os.path.isdir(source_dir):
        print(f"Warning: Source directory missing, skipped: {source_dir}")
        continue

    os.makedirs(dest_dir, exist_ok=True)
    img_files = sorted(glob(os.path.join(source_dir, '*.jpg')))

    for img_file in img_files:
        dst = os.path.join(dest_dir, os.path.basename(img_file))

        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(img_file):
            continue
        shutil.copy(img_file, dst)

# Summary
print(f'Total training images: {len(train_image_files)}')
print(f'Total testing images: {len(test_image_files)}')


## Pre-process the data: create the images and labels folders for both training and testing set

In [ ]:
CLASS_MAPPING = {
    'car': 0,
    'bus': 1,
    'van': 2,
}

def process_detrac_annotations(annotations_dir, images_path, labels_path, dataset_type):
    """
    Process DETRAC annotations and create YOLO format labels
    
    Args:
        annotations_dir: Directory containing XML annotation files
        images_path: Output directory for images (with sequence folders)
        labels_path: Output directory for labels (with sequence folders)
        dataset_type: 'train' or 'test'
    """
    os.makedirs(images_path, exist_ok=True)
    os.makedirs(labels_path, exist_ok=True)

    processed_count = 0
    
    for xml_file in tqdm(os.listdir(annotations_dir), desc=f"Processing {dataset_type} sequences"):
        if not xml_file.endswith('.xml'):
            continue
        
        sequence_name = os.path.splitext(xml_file)[0]

        # CHG 1: lire depuis le split du code 1
        images_root = train_output_dir if dataset_type == 'train' else test_output_dir
        image_dir = os.path.join(images_root, sequence_name)
        
        if not os.path.isdir(image_dir):
            continue

        tree = ET.parse(os.path.join(annotations_dir, xml_file))
        root = tree.getroot()

        img_width, img_height = None, None
        try:
            first_image_path = os.path.join(image_dir, sorted(os.listdir(image_dir))[0])
            with Image.open(first_image_path) as img:
                img_width, img_height = img.size
        except Exception as e:
            continue
        
        # Create sequence-specific directories for images and labels
        seq_images_path = os.path.join(images_path, sequence_name)
        seq_labels_path = os.path.join(labels_path, sequence_name)
        os.makedirs(seq_images_path, exist_ok=True)
        os.makedirs(seq_labels_path, exist_ok=True)
            
        frames = root.findall('frame')
        for frame in frames:
            frame_num = int(frame.get('num'))
            image_filename = f"img{frame_num:05d}.jpg"
            label_filename = f"img{frame_num:05d}.txt"
            
            source_image_path = os.path.join(image_dir, image_filename)
            dest_image_path = os.path.join(seq_images_path, image_filename)
            dest_label_path = os.path.join(seq_labels_path, label_filename)

            if not os.path.exists(source_image_path):
                continue

            # CHG 2: skip copy si déjà copié
            if not (os.path.exists(dest_image_path) and os.path.getsize(dest_image_path) == os.path.getsize(source_image_path)):
                shutil.copy(source_image_path, dest_image_path)

            # CHG 3: skip si label déjà généré
            if os.path.exists(dest_label_path):
                processed_count += 1
                continue
            
            yolo_annotations = []
            target_list = frame.find('target_list')
            if target_list is not None:
                for target in target_list.findall('target'):
                    box = target.find('box')
                    attribute = target.find('attribute')
                    
                    vehicle_type = attribute.get('vehicle_type')
                    if vehicle_type not in CLASS_MAPPING:
                        continue
                    
                    class_id = CLASS_MAPPING[vehicle_type]
                    xmin = float(box.get('left'))
                    ymin = float(box.get('top'))
                    width = float(box.get('width'))
                    height = float(box.get('height'))
                    
                    x_center = (xmin + width / 2) / img_width
                    y_center = (ymin + height / 2) / img_height
                    w_norm = width / img_width
                    h_norm = height / img_height
                    
                    yolo_annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

            with open(dest_label_path, 'w') as f:
                f.write('\n'.join(yolo_annotations))
            processed_count += 1

    print(f"\n{dataset_type.capitalize()} - Total images and labels successfully processed: {processed_count}")
    return processed_count

# Process training set
train_images_path = os.path.join(OUTPUT_PATH, 'train', 'images')
train_labels_path = os.path.join(OUTPUT_PATH, 'train', 'labels')
train_count = process_detrac_annotations(
    TRAIN_ANNOTATIONS_DIR, 
    train_images_path, 
    train_labels_path, 
    'train'
)

# Process testing set
test_images_path = os.path.join(OUTPUT_PATH, 'test', 'images')
test_labels_path = os.path.join(OUTPUT_PATH, 'test', 'labels')
test_count = process_detrac_annotations(
    TEST_ANNOTATIONS_DIR, 
    test_images_path, 
    test_labels_path, 
    'test'
)

print(f"\n✅ Conversion complete!")
print(f"Training: {train_count} images with labels")
print(f"Testing: {test_count} images with labels")
